**Por que estatística é importante em machine learning?**

Um modelo aprende padrões a partir de dados, isso difere da programação comum em que o humano insere REGRAS DETERMINADAS padrões para um programa realizar uma tarefa.

A estatística é ferramenta para:
- Para entender os dados antes
- Percebermos e resolvermos problemas como valores estranhos ou extremos, dados faltantes ou mal registrados
- Comparar previsão com o valor real
- Analisar o quanto determinado resultado é confiável
- Evitar conclusões preciptadas sobre o que modelo aprendeu

**Analise exploratória e validação estatisitica**

Por que?
Conhecer os dados antes e fazer uso da estatística nos permite uma orientação melhor sobre o modelo
Modelos complexos para problemas simples ou modelos simples para problemas complexos geram perda de tempo, recurso e consequentemente valor no desenvolvimento de um projeto de IA

**Aplicação**
Vamos explorar a base de dados, *base_imoveis.csv*
Primeiro começamos lendo a base para criar o dataframe, realizar contagens, explorar a base.
Depois vamos restringir o dataframe para apenas algumas linhas e realizar a validação estatística afim de nos orientarmos sobre o modelo de ML
Plot de gráficos pra auxiliar a visualização
Por fim teremos a justificativa de uso do modelo

**LEITURA E RECONHECIMENTO**

In [ ]:
import pandas as pd

# 1. Carregar a base de dados
df = pd.read_csv(r'C:\Users\marciohenrique\Documents\Projetos\github\ml_stat\bases_dados\base_imoveis.csv'\
                 ,encoding='utf-8') 

# 2. Visualizar as primeiras linhas
display(df.head())

# 3. Verificar o tamanho da base (linhas, colunas)
print(df.shape )

# 4. Mapear os tipos de colunas e dados ausentes

print(df.info())

# 5. Estatística descritiva inicial (para variáveis numéricas)
print("\n--- Resumo Estatístico Inicial ---")
display(df.describe())


**Perguntas Inciais**

- Quais são dimensões da base?
- Quantas features existem?
- Quais são categoricas e quais contínuas?

**Missing Values**
- Existem dados ausentes? Se sim, em quais colunas?
- Como esses dados ausentes afetam o treinamento de um modelo de ML?
-  Para discussão: O que podemos fazer com esses dados ausentes:
    -   Inserindo artificalmente a média ou mediana quais consequencias para o modelo?
    - Exclusões quais seriam as consequências para o modelo?

**Resumo Estatístico**
- Qual maior e menor preço?
- Qual é média de idade dos imóveis?
- Qual faixa percentual encontra-se a maior metragem dos imóveis
- Qual é a mediana do preço de venda?
- Qual percentual custa menos de 665k?

**Dispersão**
Percentuais altos indicam alta **HETEROGENIDADE**
- Vamos calcular a relação entre o desvio padrao e a média do preço e tirar conclusões

In [ ]:
df['preco_venda'].std()
df['preco_venda'].mean()
dispersao = df['preco_venda'].std() / df['preco_venda'].mean()*100
print(f"\n--- Dispersão (Desvio Padrão / Média) ---\nDispersão: {dispersao:.2f}%")

**Conclusão mateática sobre a base**
**É altamente heteorgenea**
- Vamos restringir o df e focar em uma ou 2 features
- vamos contabilizar os nulos e entender o percentual de nulos da base
- apartamentos e casas tem tamanhos geralmente diferentes o que pode influenciar ou não no preço...
- assim vamos partir os dados entre casas e aprtamentos, eliminar os valores nulos e analisar os resultados comparando segmentados com nao segmentados

In [ ]:
total_nulos = df.isnull().sum().sum()
percentual_nulos = (total_nulos / df.size) * 100

print(f"\n--- Percentual de Valores Nulos na Base ---\nTotal de valores nulos: {total_nulos}\nPercentual de valores nulos: {percentual_nulos:.2f}%")

print(f"\n--- Percentual de nulos por coluna---\n{((df.isnull().sum() / len(df)) * 100)}")


In [ ]:
#dataframses segmentados (removendo nulos do preço)
df_apartamentos = df[df['tipo'] == 'Apartamento'].dropna(subset=['preco_venda'])
df_casas = df[df['tipo'] == 'Casa'].dropna(subset=['preco_venda'])

In [ ]:
print("=== APARTAMENTOS ===")
print (df_apartamentos.describe())

In [ ]:
print("=== CASAS ===")
print (df_casas.describe())

In [ ]:
#teste de normalidade (Shapiro-Wilk) para verificar se os preços de venda seguem uma distribuição normal
from scipy import stats
_, p_ap = stats.shapiro(df_apartamentos['preco_venda'].sample(min(len(df_apartamentos), 1000), random_state=42))
_, p_ca = stats.shapiro(df_casas['preco_venda'].sample(min(len(df_casas), 1000), random_state=42))


In [ ]:
print("\n--- Teste de Normalidade (Amostra de 1000 imóveis) ---")
print(f"Apartamentos - p-valor: {p_ap:.4f} -> " + ("É Normal" if p_ap > 0.05 else "NÃO é Normal"))
print(f"Casas        - p-valor: {p_ca:.4f} -> " + ("É Normal" if p_ca > 0.05 else "NÃO é Normal"))

**Teste Shapiro Wilk**

O teste é indicador de normalidade, ou seja que a distribuição dos dados é equalitaria no histograma;
Ou em outras palavras se o resultado é normal, então **podemos confiar na média** não há muitas distorções e o modelo de machine learning\
de regressão linear simples pode ser usado.
**p valor não normal** Aí siginifca que há outiliers sigfificativos e que o modelo de arvores de decisão é o mais adequado, como random forest